In [ ]:
# Coisas a Melhorar: 

# 1) Passar tudo para inglês

# 2) Encontrar um jeito de adquirir os parâmetros simulados do paraquedas

# 3) Problema de travamento em Monte Carlo (no meu note - Ian)

# 4) Atualizar dados com as novas versões do OR (v2.11)

## Instruções ⚠️

### Para um bom uso desse código durante a LASC, siga os passos descritos abaixo, boa simulação!

### 1) Modifique parâmetros de simulação APENAS pela Caixa de Edição do Usuário, que pode ser encontrada facilmente usando o "Outline" do VsCode, na barra lateral esquerda.

### 2) Para facilitar a visualização, todas as imagens e plots de Monte Carlo serão salvas em um pdf. É possível acessá-lo na barra lateral esquerda, depois de gerá-lo.

# 

## Lib import

In [ ]:
import datetime

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import threading

from rocketpy import Environment, Flight, MonteCarlo, Rocket, SolidMotor

from rocketpy.stochastic import (
    StochasticEnvironment,
    StochasticFlight,
    StochasticNoseCone,
    StochasticRailButtons,
    StochasticParachute,
    StochasticRocket,
    StochasticSolidMotor,
    StochasticTail,
    StochasticTrapezoidalFins,
)

## Launch Site

### Parte do código onde é definido as váriaveis do local e ambiente de lançamento

In [ ]:
env = Environment(
    latitude = -21.9419,                # Latitude da LASC
    longitude= -48.9531,                # Longitude da LASC
    timezone = 'America/Sao_Paulo',     # Fuso horário
    datum="WGS84"                       # Fonte das coordenadas (Padrão mundial do GPS)
)

DATETIME = datetime.datetime.fromisoformat("2026-06-23 12:00:00")       # Horário (Para verificar as condições do tempo)
env.set_date(DATETIME)


resultado = {"sucesso": False, "erro": None}

def download_gefs():                                                    # Função para obter os dados vindos do Ensemble - GEFS
    try:
        env.set_atmospheric_model(type="Ensemble", file="GEFS")         # Modelo atmosférico (Ensemble é a melhor escolha para Monte Carlo)
        resultado["sucesso"] = True
    except Exception as e:
        resultado["erro"] = e

thread = threading.Thread(target=download_gefs)                         # Como GEFS possui arquivos pesados, é necessário ter um modelo
thread.start()                                                          # atimosférico de backup, caso não baixe o Ensemble corretamente
thread.join(timeout=10)  # espera no máximo x segundos 

if thread.is_alive():
    print("GEFS não retornou — usando ECMWF como backup")
    env.set_atmospheric_model(type="Windy", file="ECMWF")
elif resultado["sucesso"]:
    print("GEFS carregado com sucesso")
else:
    print(f"GEFS retornou com erro: {resultado['erro']} — usando GFS")
    env.set_atmospheric_model(type="Windy", file="ECMWF")


env.set_topographic_profile(type="NASADEM_HGT", file="NASADEM_NC_s22w049.nc", dictionary="netCDF4", crs=None)       # Modelo topográfico

elevation = env.get_elevation_from_topographic_profile(env.latitude, env.longitude)

env.set_elevation(elevation)


env.all_info()      # Visualizar todas as informações

### Elevação do solo coletada pela API da NASA

## Stockhastic Launch Site

### Definição do desvio nos parâmetros do ambiente de lançamento para a simulação de Monte Carlo

In [ ]:
if env.atmospheric_model_type == "Ensemble":                    # Faz a checagem do modelo atmosférico para definir o Ensemble
    ensemble_member=list(range(env.num_ensemble_members))       # Lista de previsões prováveis para o local
else:
    ensemble_member=None


stochastic_env = StochasticEnvironment(
    environment=env,
    
    ensemble_member=ensemble_member,

    wind_velocity_x_factor=(1.0, 0.1),      # Fator multiplicativo para a velocidade do vento na direção x, desvio padrão
    wind_velocity_y_factor=(1.0, 0.1),      # Fator multiplicativo para a velocidade do vento na direção y, desvio padrão
)

stochastic_env.visualize_attributes()

## Atlas Motor

### Definição do motor do Atlas

In [ ]:
Proton = SolidMotor(
    thrust_source="Proton.eng",       # Arquivo .eng do motor
    dry_mass = 15.652,               
    dry_inertia=(0.0238, 1.065, 1.065),
    nozzle_radius= 36.03 / 1000,
    grain_number= 6,
    grain_density= 1750.00622798,
    grain_outer_radius= 45 / 1000,
    grain_initial_inner_radius= 19.05 / 1000,
    grain_initial_height= 140 / 1000,
    grain_separation= 12 / 1000,
    grains_center_of_mass_position= 545.448 / 1000,      # Tirado do Fusion (Tamanho do motor - CM com apenas os grãos -- Origem está invertida no CAD
    center_of_dry_mass_position= 521.402 / 1000,         # Tirado do Fusion (Tamanho motor - CM) -- Origem está invertida no CAD
    nozzle_position= 0 /1000,  # Posição do fim do Nozzle
    throat_radius= 13.5 / 1000,
    coordinate_system_orientation="nozzle_to_combustion_chamber",
)

Proton.info()

#### Versão teórica do Proton

## Stockhastic Atlas Motor

### Definição dos desvios nos parâmetros do motor do Atlas para simulação de Monte Carlo

In [ ]:
stochastic_Motor = StochasticSolidMotor(
    solid_motor=Proton,
    burn_start_time=(0, 0.01, "binomial"), # Média, Desvio-Padrão, tipo de distribuição
    grains_center_of_mass_position= 0,
    grain_density= 0,
    grain_separation= 0 / 1000,
    grain_initial_height= 0 / 1000,
    grain_initial_inner_radius= 0 / 1000,
    grain_outer_radius= 0 / 1000,
    total_impulse=(9588, 200),
    throat_radius= 0 / 1000,  
    nozzle_radius= 0 / 1000, 
    nozzle_position= 0,
)
stochastic_Motor.visualize_attributes()

#### Desvios confirmados pelo setor de propulsão

## Rocket structure

### Definição de toda a parte estrutural do foguete Atlas

In [ ]:
Atlas = Rocket(
    radius= 76/1000,
    mass= 13896/1000,
    inertia=(12.66, 0.0697, 12.66), # Tirado do Assembly
    power_off_drag="Atlas_CD_Power-Off.csv",          # Tirado do RaasAero
    power_on_drag="Atlas_CD_Power-On.csv",           # Tirado do RaasAero
    center_of_mass_without_motor= 1268/1000,
    coordinate_system_orientation="nose_to_tail",
)

Nosecone = Atlas.add_nose(
    length= 850/1000, kind="Von Karman", position=0
)

Fins = Atlas.add_trapezoidal_fins(      # Verificar se, com esses dados, a geometria é válida
    n=4,
    root_chord = 290/1000,
    tip_chord = 50/1000,
    span = 151.1/1000,
    position = 2358/1000,
    cant_angle = 0,
    sweep_length = 230/1000,
)


Atlas.add_motor(Proton, position= 2646/1000) 


boattail = Atlas.add_tail(
    top_radius=76/1000, bottom_radius=64.5/1000, length=300/1000, position=2348/1000, name="Boattail Cônico",
)

rail_buttons = Atlas.set_rail_buttons(
    upper_button_position= 1540/1000,
    lower_button_position= 2320/1000,
    angular_position=45,
)

main_parachute = Atlas.add_parachute( 
    name="Main Parachute Atlas",
    cd_s=5.193,     # cd: 0,9 X s: 5,77 m^2 (REC ainda vai atualizar esses valores)
    trigger= 425,  # altitude de ejeção (m)
    sampling_rate=50,   # taxa de atualização do barômetro em Hz (checado pela aviônica)
    lag=0.118,        # Período de abertura do paraquedas (calculado por Rec)
)

drogue_parachute = Atlas.add_parachute( 
    name="Drogue Parachute Atlas",
    cd_s=0.452,     # cd: 0,4 X s: 1,13 m^2 (REC ainda vai atualizar esses valores)
    trigger= "apogee",  # altitude de ejeção (m)
    sampling_rate=50,   # taxa de atualização do barômetro em Hz (checado pela aviônica)
    lag=0.118,        # Período de abertura do paraquedas (calculado por Rec)
)

#### Se formos usar o Monte Carlo, não é possível colocar aletas com formato personalizado (por pontos)

## Stockhastic rocket structure

### Definição dos desvios nos parâmetros estruturais do Atlas para simulação de Monte Carlo

In [ ]:
stochastic_Atlas = StochasticRocket(
    rocket=Atlas,
    radius= 0.005, # Ir medir no lab
    mass=(13896/1000, 200/1000, "normal"), # Média, Desvio-Padrão, tipo de distribuição
    inertia_11=(12.66, 0.001),
    inertia_22=0.0001,
    inertia_33=0.001,
    center_of_mass_without_motor=0.005,
    power_off_drag_factor=(1, 0.05),  # Multiplier for rocket's drag curve. Usually has a mean value of 1 
                                            # and a uncertainty of 5% to 10%

    power_on_drag_factor=(1, 0.05),   # Multiplier for rocket's drag curve. Usually has a mean value of 1 
                                            # and a uncertainty of 5% to 10%
)

stochastic_Nosecone = StochasticNoseCone(
    nosecone=Nosecone,
    length=0.001,
)

stochastic_Fins = StochasticTrapezoidalFins(
    trapezoidal_fins=Fins,
    root_chord=0.0005,
    tip_chord=0.0005,
    span=0.0005,
)

stochastic_boattail = StochasticTail(
    tail=boattail,
    top_radius=0.0005,
    bottom_radius=0.0005,
    length=0.001,
)

stochastic_rail_buttons = StochasticRailButtons(
    rail_buttons=rail_buttons, buttons_distance=0.0005
)

stochastic_main_parachute = StochasticParachute(
    parachute=main_parachute,
    cd_s=0.2,   # Recomendado por Rec
    lag=0.032,  # Calculado por Rec
)

stochastic_drogue_parachute = StochasticParachute(
    parachute=drogue_parachute,
    cd_s=0.2,   # Recomendado por Rec
    lag=0.032,  # Calculado por Rec
)


stochastic_Atlas.add_motor(stochastic_Motor, position=0.001)
stochastic_Atlas.add_nose(stochastic_Nosecone, position=(0, 0.001))
stochastic_Atlas.add_trapezoidal_fins(stochastic_Fins, position=(0.001, "normal"))
stochastic_Atlas.add_tail(stochastic_boattail)
stochastic_Atlas.set_rail_buttons(stochastic_rail_buttons, lower_button_position=(0.0005, "normal"))
stochastic_Atlas.add_parachute(stochastic_main_parachute)
stochastic_Atlas.add_parachute(stochastic_drogue_parachute)

stochastic_Atlas.visualize_attributes()
#stochastic_Fins.visualize_attributes()
#stochastic_coifa.visualize_attributes()

#### Desvios confirmados pelo setor de Estruturas

## Flight Conditions

### Definição das condições de voo na base de lançamento

In [ ]:
flightStage = Flight(
    rocket=Atlas,
    environment=env,
    rail_length=6,  # meters
    inclination=80, # degrees
    heading=90,     # degrees
)

## Stockhastic flight conditions

### Definição dos desvios nas condições de voo na base de lançamento para simulação de Monte Carlo 

In [ ]:
stochastic_flight = StochasticFlight(
    flight=flightStage,
    rail_length=(6, 0.01),
    inclination=(80, 1),  # ângulo, desvio-padrão
    heading=(90, 2),     # ângulo, desvio-padrão
)

stochastic_flight.visualize_attributes()

<h1 style="color: orange; font-size: xxx-large">CAIXA DE EDIÇÃO DO USUÁRIO ⚙️</h1>


In [ ]:
# Aqui está a função que permite o usuário modificar alguma informação importante do código, caso queira

import importlib
import modular_code_edition as mce
importlib.reload(mce)


mce.modifica(

# Digite na linha abaixo suas modificações:


)


# Aqui está uma lista de todas as modificações possíveis para o dia da competição:

    # Ambiente
        # data_hora="2026-0x-xx xx:xx:xx"
        # latitude
        # longitude
        # timezone="Continente/Cidade"

    # Motor
        # densidade
        # separacao_entre_graos

    # Estrutura
        # massa
        # Ixx
        # Iyy
        # cm_sem_motor

    # Lançamento
        # tam_haste
        # inclinacao
        # direcao

## Visual configuration

### Checagem visual de componentes do foguete definida no código 

In [ ]:
Proton.draw()
Proton.draw(filename="1Proton.png")

Fins.draw()
Fins.draw(filename="1Fins.png")

Atlas.draw()
Atlas.draw(filename="1Atlas.png")


## Monte Carlo Simulation

In [ ]:
Simulation = MonteCarlo(
    filename="Resultados_Monte-Carlo",
    environment=stochastic_env,
    rocket=stochastic_Atlas,
    flight=stochastic_flight,
)

Simulation.simulate(
    number_of_simulations=100,      # Rodar 1000
    append=False,
    include_function_data=False,
    parallel=True,
    n_workers=None,
)

## Results

### Valores das variáveis de voo (Comum)

In [ ]:
Simulation.prints.all()

flightStage.prints.maximum_values()

### Salvando os resultados em dicionário

In [ ]:
simulation_general_results = []

simulation_results = {
    "out_of_rail_time": [],
    "out_of_rail_velocity": [],
    "apogee_time": [],
    "apogee": [],
    "apogee_x": [],
    "apogee_y": [],
    "t_final": [],
    "x_impact": [],
    "y_impact": [],
    "impact_velocity": [],
    "initial_stability_margin": [],
    "out_of_rail_stability_margin": [],
    "max_mach_number": [],
    "frontal_surface_wind": [],
    "lateral_surface_wind": [],
    "index": [],
    "drogue_triggerTime": [],           # Aparentemente não existe como resultado da simulação (Precisa de outro rolo pra funcionar)
    "drogue_inflated_velocity": [],     # Aparentemente não existe como resultado da simulação (Precisa de outro rolo pra funcionar)
}

simulation_output_file = open(str("Resultados_Monte-Carlo") + ".outputs.txt", "r+")

# Read each line of the file and convert to dict
for line in simulation_output_file:
    # Skip comments lines
    if line[0] != "{":
        continue
    # Eval results and store them
    flight_result = eval(line)
    simulation_general_results.append(flight_result)
    for parameter_key, parameter_value in flight_result.items():
        simulation_results[parameter_key].append(parameter_value)

# Close data file
simulation_output_file.close()

# Print number of flights simulated
N = len(simulation_general_results)

### Apogeu resultante (Monte Carlo)

In [ ]:
print(
    f"Apogee Altitude - Mean Value: {np.mean(simulation_results['apogee']):0.3f} m"
)
print(
    f"Apogee Altitude - Standard Deviation: {np.std(simulation_results['apogee']):0.3f} m"
)

plt.figure()
plt.hist(simulation_results["apogee"], bins=int(N**0.5))
plt.title("Apogee Altitude")
plt.xlabel("Altitude (m)")
plt.ylabel("Number of Occurences")
plt.show()

# ---------------------------------------------------

print(
    f"Apogee Time - Mean Value: {np.mean(simulation_results['apogee_time']):0.3f} s"
)
print(
    f"Apogee Time - Standard Deviation: {np.std(simulation_results['apogee_time']):0.3f} s"
)

plt.figure()
plt.hist(simulation_results["apogee_time"], bins=int(N**0.5))
plt.title("Apogee Time")
plt.xlabel("Time (s)")
plt.ylabel("Number of Occurences")
plt.show()

### Margem de estabilidade (Comum + Monte Carlo)

In [ ]:
Atlas.plots.static_margin()

# ---------------------------------------

print(
    f"Stability Margin - Mean Value: {np.mean(simulation_results['initial_stability_margin']):0.3f} cal"
)
print(
    f"Stability Margin - Standard Deviation: {np.std(simulation_results['initial_stability_margin']):0.3f} cal"
)

plt.figure()
counts, edges, patches = plt.hist(simulation_results["initial_stability_margin"], bins=int(N**0.5))
plt.title("Stability Margin")
plt.xlabel("Cal")
plt.ylabel("Number of Occurences")
plt.xticks(edges, labels=[f"{e:.2f}" for e in edges])
plt.tight_layout()
plt.show()

# ---------------------------------------

# Adicionar alguma outra margem de estabilidade?

### Velocidades de vento / Velocidade de impacto no solo (Monte Carlo)

In [ ]:
print(
    f"Frontal Surface Wind - Mean Value: {np.mean(simulation_results['frontal_surface_wind']):0.3f} m/s"
)
print(
    f"Frontal Surface Wind - Standard Deviation: {np.std(simulation_results['frontal_surface_wind']):0.3f} m/s"
)

plt.figure()
plt.hist(simulation_results["frontal_surface_wind"], bins=int(N**0.5))
plt.title("Frontal Surface Wind")
plt.xlabel("Velocity (m/s)")
plt.ylabel("Number of Occurences")
plt.show()

# ----------------------------------------------------------------

print(
    f"Lateral Surface Wind - Mean Value: {np.mean(simulation_results['lateral_surface_wind']):0.3f} s"
)
print(
    f"Lateral Surface Wind - Standard Deviation: {np.std(simulation_results['lateral_surface_wind']):0.3f} s"
)

plt.figure()
plt.hist(simulation_results["lateral_surface_wind"], bins=int(N**0.5))
plt.title("Lateral Surface Wind")
plt.xlabel("Velocity (m/s)")
plt.ylabel("Number of Occurences")
plt.show()

# ----------------------------------------------------------------

print(
    f"Impact Velocity - Mean Value: {np.mean(simulation_results['impact_velocity']):0.3f} m/s"
)
print(
    f"Impact Velocity - Standard Deviation: {np.std(simulation_results['impact_velocity']):0.3f} m/s"
)

plt.figure()
plt.hist(simulation_results["impact_velocity"], bins=int(N**0.5))
plt.title("Impact Velocity")
plt.xlabel("Velocity (m/s)")
plt.ylabel("Number of Occurences")
plt.show()


### Velocidade (Comum), Mach Max e Tempo de voo (Monte Carlo)

In [ ]:
flightStage.speed.plot(0, flightStage.apogee_time)

# ----------------------------------------------------------------

print(
    f"Flight time - Mean Value: {np.mean(simulation_results['t_final']):0.3f} s"
)
print(
    f"Flight time - Standard Deviation: {np.std(simulation_results['t_final']):0.3f} s"
)

plt.figure()
plt.hist(simulation_results["t_final"], bins=int(N**0.5))
plt.title("Flight time")
plt.xlabel("Time (s)")
plt.ylabel("Number of Occurences")
plt.show()

# ----------------------------------------------------------------

print(
    f"Max Mach Number - Mean Value: {np.mean(simulation_results['max_mach_number']):0.3f} "
)
print(
    f"Max Mach Number - Standard Deviation: {np.std(simulation_results['max_mach_number']):0.3f} "
)

plt.figure()
plt.hist(simulation_results["max_mach_number"], bins=int(N**0.5))
plt.title("Max Mach Number")
plt.xlabel("Mach")
plt.ylabel("Number of Occurences")
plt.show()

### Ativação do Drogue e velocidade no momento de abertura do Drogue (Monte Carlo)

#### Não sei se vai funcionar

In [ ]:
print(
    f"Drogue Parachute Trigger Time - Mean Value: {np.mean(simulation_results['drogue_triggerTime']):0.3f} s"
)
print(
    f"Drogue Parachute Trigger Time - Standard Deviation: {np.std(simulation_results['drogue_triggerTime']):0.3f} s"
)

plt.figure()
plt.hist(simulation_results["drogue_triggerTime"], bins=int(N**0.5))
plt.title("Drogue Parachute Trigger Time")
plt.xlabel("Time (s)")
plt.ylabel("Number of Occurences")
plt.show()

# ----------------------------------------------------------------

print(
    f"Drogue Parachute Fully Inflated Velocity - Mean Value: {np.mean(simulation_results['drogue_inflated_velocity']):0.3f} m/s"
)
print(
    f"Drogue Parachute Fully Inflated Velocity - Standard Deviation: {np.std(simulation_results['drogue_inflated_velocity']):0.3f} m/s"
)

plt.figure()
plt.hist(simulation_results["drogue_inflated_velocity"], bins=int(N**0.5))
plt.title("Drogue Parachute Fully Inflated Velocity")
plt.xlabel("Velocity m/s)")
plt.ylabel("Number of Occurences")
plt.show()

### Raio de aterrisagem provável (Monte Carlo)

In [ ]:
Simulation.plots.ellipses(xlim=(-9000, 9000), ylim=(-9000, 9000)) # Visualização do raio de aterrisagem
Simulation.plots.ellipses(save=True)

ax = plt.gca()
ax.xaxis.set_major_locator(plt.MultipleLocator(500))    # Espaçamento de 500m nos eixos
ax.yaxis.set_major_locator(plt.MultipleLocator(500))
plt.savefig("Resultados_Monte-Carlo.png")
plt.show()

#image="Launch Site 2km_x_2km.png" é inserido no argumento "ellipses" para colocar background

### Trajetória de voo 3D

In [ ]:
flightStage.plots.trajectory_3d()

flightStage.plots.trajectory_3d(filename="1Trajectory_Stage.png")


### Salvando todos os plots em um PDF

In [ ]:
plt.ioff()

with PdfPages('Plots_de_Resultados.pdf') as pdf:

    plt.figure()
    img_P = plt.imread('1Proton.png')
    plt.imshow(img_P)
    plt.axis('off')
    plt.title('Proton')
    pdf.savefig()
    plt.close()

    plt.figure()
    img_F = plt.imread('1Fins.png')
    plt.imshow(img_F)
    plt.axis('off')
    plt.title('Fins')
    pdf.savefig()
    plt.close()

    plt.figure()
    img_A = plt.imread('1Atlas.png')
    plt.imshow(img_A)
    plt.axis('off')
    plt.title('Atlas')
    pdf.savefig()
    plt.close()

    plt.figure()
    plt.hist(simulation_results["apogee"], bins=int(N**0.5))
    plt.title("Apogee Altitude")
    plt.xlabel("Altitude (m)")
    plt.ylabel("Number of Occurences")
    pdf.savefig()
    plt.close()

    plt.figure()
    plt.hist(simulation_results["apogee_time"], bins=int(N**0.5))
    plt.title("Apogee Time")
    plt.xlabel("Time (s)")
    plt.ylabel("Number of Occurences")
    pdf.savefig()
    plt.close()

    plt.figure()
    counts, edges, patches = plt.hist(simulation_results["initial_stability_margin"], bins=int(N**0.5))
    plt.title("Stability Margin")
    plt.xlabel("Cal")
    plt.ylabel("Number of Occurences")
    plt.xticks(edges, labels=[f"{e:.2f}" for e in edges])
    plt.tight_layout()
    pdf.savefig()
    plt.close()

    plt.figure()
    plt.hist(simulation_results["frontal_surface_wind"], bins=int(N**0.5))
    plt.title("Frontal Surface Wind")
    plt.xlabel("Velocity (m/s)")
    plt.ylabel("Number of Occurences")
    pdf.savefig()
    plt.close()

    plt.figure()
    plt.hist(simulation_results["lateral_surface_wind"], bins=int(N**0.5))
    plt.title("Lateral Surface Wind")
    plt.xlabel("Velocity (m/s)")
    plt.ylabel("Number of Occurences")
    pdf.savefig()
    plt.close()

    plt.figure()
    plt.hist(simulation_results["impact_velocity"], bins=int(N**0.5))
    plt.title("Impact Velocity")
    plt.xlabel("Velocity (m/s)")
    plt.ylabel("Number of Occurences")
    pdf.savefig()
    plt.close()

    plt.figure()
    plt.hist(simulation_results["t_final"], bins=int(N**0.5))
    plt.title("Flight time")
    plt.xlabel("Time (s)")
    plt.ylabel("Number of Occurences")
    pdf.savefig()
    plt.close()

    plt.figure()
    plt.hist(simulation_results["max_mach_number"], bins=int(N**0.5))
    plt.title("Max Mach Number")
    plt.xlabel("Mach")
    plt.ylabel("Number of Occurences")
    pdf.savefig()
    plt.close()

    plt.figure()
    plt.hist(simulation_results["drogue_triggerTime"], bins=int(N**0.5))            # Resolver o salvamento do parâmetro pra funcionar
    plt.title("Drogue Parachute Trigger Time")
    plt.xlabel("Time (s)")
    plt.ylabel("Number of Occurences")
    pdf.savefig()
    plt.close()

    plt.figure()
    plt.hist(simulation_results["drogue_inflated_velocity"], bins=int(N**0.5))      # Resolver o salvamento do parâmetro pra funcionar
    plt.title("Drogue Parachute Fully Inflated Velocity")
    plt.xlabel("Velocity m/s)")
    plt.ylabel("Number of Occurences")
    pdf.savefig()
    plt.close()

    plt.figure()
    img_E = plt.imread('Resultados_Monte-Carlo.png')
    plt.imshow(img_E)
    plt.axis('off')
    plt.title('Raio de Aterrissagem')
    pdf.savefig()
    plt.close()

    plt.figure()
    img_P = plt.imread('1Trajectory_Stage.png')
    plt.imshow(img_P)
    plt.axis('off')
    plt.title('Trajetória 3D')
    pdf.savefig()
    plt.close()


In [ ]:
flightStage.export_kml(
    file_name = "Atlas.kml",
    extrude = True,
    altitude_mode = "relativetoground",
)